### DEVELOPMENT

In [0]:
# Import packages and functions needed 
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType
from pyspark.sql.types import StringType


# funcion que hace explode de arrays
def explode_arrays(df):
    array_columns = []
    
    # busca y selecciona las columnas array
    for field in df.schema.fields:
        if isinstance(field.dataType, ArrayType):
            array_columns.append(field.name)

    # itera y explota las columnas array
    for col_name in array_columns:
        df = df.withColumn(f"exploded_{col_name}", F.explode_outer(col_name))
        
        # Verifica si el tipo de dato del campo creado exploded_ es un struct
        element_type = df.schema[f"exploded_{col_name}"].dataType
        if isinstance(element_type, StructType):
            # Si lo es, extrae los elementos o subcampos, crea 1 campo nuevo por cada elemento encontrado y elimina el campo original, o sea el array
            for subfield in element_type.fieldNames():
                df = df.withColumn(f"{col_name}_{subfield}", F.col(f"exploded_{col_name}.{subfield}"))
                df = df.drop(col_name)
        else:
            # si no es un StructType (ie, StringType o DoubleType), reemplaza el campo array original con la columna explotada
            df = df.withColumn(col_name, F.col(f"exploded_{col_name}"))
        
        # elimina la columna temporal exploded_
        df = df.drop(f"exploded_{col_name}")
    
    return df


# Función para separar el campo usando '~' y ',' como delimitadores
def split_multiple_delimiters(df, input_col, output_col):
    # Usamos regexp_replace para normalizar los delimitadores a uno solo (por ejemplo ',')
    normalized_col = F.regexp_replace(input_col, '[~,]+', ',')
    # Hacemos split del resultado normalizado
    return df.withColumn(output_col, F.split(normalized_col, ','))


# Definir la función para capitalizar la primera letra, limpiar espacios y reemplazar '_'
def clean_text(df, input_col, output_col):
    return df.withColumn(
        output_col,
        F.concat(
            F.upper(F.substring(F.trim(F.regexp_replace(F.col(input_col), '_', ' ')), 1, 1)),
            F.lower(F.substring(F.trim(F.regexp_replace(F.col(input_col), '_', ' ')), 2, 1000))
        )
    )

# funcion para extraer el contenido entre corchetes
def extract_string_content(df, input_col, output_col):
    return df.withColumn(
        output_col,
        F.when(
            F.col(input_col).rlike(r'\[.*?\]'),  # Si contiene corchetes
            F.regexp_replace(  # Remover las comillas dobles después de extraer el contenido
                F.regexp_extract(F.col(input_col), r'\[(.*?)\]', 1),
                r'"', ''  # Reemplazar todas las comillas dobles por un string vacío
            )
        ).otherwise(F.col(input_col))  # Si no tiene corchetes, dejar el valor original
    )


In [0]:
# Import tables

## bmdm table
gdm = spark.table("crm_reporting.dim_gdm_brand_profile")

# Comando que permite ver la ruta real de una tabla en databricks a partir de su metastore
# DESCRIBE DETAIL crm_reporting.dim_gdm_brand_profile;

# print(gdm.count()) #10496032
# print(len(gdm.columns))


####### FIRST SELECTION OF ATTRIBUTES WITH AT LEAST 1 NON-VALUE
df = gdm
# Count non-null values per column
agg_expr = [F.count(F.when(F.col(c).isNotNull(), c)).alias(c) for c in df.columns]

# Execute the aggregation
non_null_counts = df.agg(*agg_expr).collect()[0]

# Keep the not enterely null columns
non_completely_null_columns = [c for c in df.columns if non_null_counts[c] > 0]

# count of not-enterely null columns
# print(len(non_completely_null_columns)) ## 78

# select the founded fields
gdm_cols_filt = df.select(non_completely_null_columns)#.\
  # withColumnRenamed("created_dt",'created_date')



##### OTHER COLUMNS TO REMOVE
columns_to_remove = ['dept_store_regional','etl_batch_id','mdm_source','sys_created_by','beauty_supply_store','category','product_category']

##### Filter columns that contain the string '_dt' and the ones in columns_to_remove
columns_wt_dt = [col for col in gdm_cols_filt.columns if '_dt' not in col and col not in columns_to_remove]

# count of not-enterely null columns
# print(len(columns_wt_dt)) ## 51
#print(non_completely_null_columns)


# select the list of columns to keep
gdm_cols_filt = gdm_cols_filt.select(columns_wt_dt)
gdm_cols_filt.createOrReplaceTempView("gdm_cols_filt_vw")

In [0]:
### Merged tables
merged_gdm = spark.sql("""
select 
  upper(a.source_name) as source_name,
  lower(a.registration_sub_source) AS registration_sub_source,
  a.source_customer_id,
  -- e.channel_name, 
  --c.brand_customer_id,
  --a.last_modified_dt as created_date,
  case when a.created_dt is null then date(a.last_modified_dt)
    else date(a.created_dt) end as created_date,
  d.*
from prod_latam_catalog.crm_reporting.dim_customer a
inner join prod_latam_catalog.crm_reporting.dim_customer_bridge c
  on a.source_customer_id = c.source_customer_id 
  and a.brand_code = c.brand_code 
  and a.brand_country = c.brand_country
-- inner join expld_gdm_vw d 
inner join gdm_cols_filt_vw d
  on c.brand_mdm_id = d.brand_mdm_id 
  and c.brand_code = d.brand_code 
  and c.brand_country = d.brand_country
-- left join prod_latam_catalog.crm_reporting.dim_channel e 
--   on concat(UPPER(a.source_name),"_",UPPER(a.acq_source),"_",UPPER(a.registration_source)) = e.row_key
where upper(a.source_name) IN ('DEMANDWARE','JEBBIT','SITECORE') 
  AND (lower(a.registration_source) LIKE ('%quiz%') 
     OR lower(a.registration_source) LIKE ('%diagnos%')
     OR lower(a.registration_source) LIKE ('%website%'))
  AND lower(a.registration_sub_source) NOT IN ('cart checkout','header','')
  -- lower(a.registration_sub_source) not in ('contest',null,'registration','','Footer')
  --AND to_date(a.created_dt) >= '2023-01-01'
group by all
order by registration_sub_source,brand_code, brand_country

""")

# print(merged_gdm.count()) #303918

In [0]:
# call the expand array function twice
expld_gdm_1 = explode_arrays(merged_gdm) # explode first array levels
expld_gdm_1 = explode_arrays(expld_gdm_1) # explode second array levels

# print(expld_gdm.count()) # 433098
# print(len(expld_gdm.columns)) ## 81
# display(expld_gdm.limit(1))
expld_gdm_1.createOrReplaceTempView("expld_gdm_1_vw")
  # print(expld_gdm_1.count()) # 433098

In [0]:
expld_gdm = spark.sql("""
with dim_customer_1 as (
  select distinct
    brand_country,
    brand_code,
    source_customer_id,
    case when created_dt is null then date(last_modified_dt)
      else date(created_dt) end as created_date,
    concat(UPPER(source_name),"_",UPPER(acq_source),"_",UPPER(registration_source)) as row_key
  from prod_latam_catalog.crm_reporting.dim_customer
),

dim_customer_date as	(
  select distinct
    brand_country,
    brand_code,
    source_customer_id,
    FIRST_VALUE(created_date) 
      OVER (PARTITION BY brand_country,brand_code,source_customer_id ORDER BY created_date asc) AS created_date
  from dim_customer_1
),

dim_customer_acq_channel as	(
  select distinct
    brand_country,
    brand_code,
    source_customer_id,
    FIRST_VALUE(row_key) 
      OVER (PARTITION BY brand_country,brand_code,source_customer_id ORDER BY created_date asc) AS row_key
  from dim_customer_1
),

dim_channel as (
  select distinct
    row_key,
    channel_name
  from prod_latam_catalog.crm_reporting.dim_channel
)

select
  a.*,
  e.channel_name,
  case when b.created_date is null then null
      else 'New' end as new_client_key,
  d.row_key
from expld_gdm_1_vw a
left join dim_customer_date b
  on a.brand_country = b.brand_country
  and a.brand_code = b.brand_code 
  and a.source_customer_id = b.source_customer_id 
  and a.created_date = b.created_date
left join dim_customer_acq_channel d
  on a.brand_country = d.brand_country
  and a.brand_code = d.brand_code 
  and a.source_customer_id = d.source_customer_id 
left join dim_channel e 
  on d.row_key = e.row_key
""")

expld_gdm.createOrReplaceTempView("expld_gdm_vw")
# print(expld_gdm.count()) #301670

In [0]:
# SECOND SELECTION OF ATTRIBUTES WITH AT LEAST 1 NON-VALUE

df = expld_gdm

# Count non-null values per column
agg_expr = [F.count(F.when(F.col(c).isNotNull(), c)).alias(c) for c in df.columns]

# Execute the aggregation
non_null_counts = df.agg(*agg_expr).collect()[0]

# Keep the not enterely null columns
non_completely_null_columns = [c for c in df.columns if non_null_counts[c] > 0]

# count of not-enterely null columns
# print(len(non_completely_null_columns)) ## 47

# select the list of columns to keep
def_gdm_v1 = df.select(non_completely_null_columns).\
  withColumn("created_date", F.to_date("created_date")).\
  withColumnRenamed("skin_type_skin_type","skin_type").\
  withColumnRenamed("hair_routine_heating_heating_tool","heating_tool").\
  withColumnRenamed("skin_sensitivity_skin_sensitivity","skin_sensitivity").\
  withColumnRenamed("fragrance_routine_perfume_moment","perfume_moment").\
  withColumnRenamed("concern_improvement_goal_concern","improvement_goal_concern").\
  withColumnRenamed("channel_preference_product_purchase_channel","purchase_channel").\
  withColumnRenamed("analysis_analysis_concern_concern","analysis_concern").\
  withColumnRenamed("analysis_analysis_concern_concern_type","concern_type").\
  withColumnRenamed("analysis_analysis_concern_zone","concern_zone").\
  withColumnRenamed("analysis_analysis_type","analysis_type").\
  drop('purchase_channel','row_key')
  # drop('purchase_channel','source_customer_id')

# limpiamos valores con la estructura: Concern_type="texture uniformity";zone=["whole face", "cheek"]
def_gdm_v1 = extract_string_content(def_gdm_v1, "concern_zone", "concern_zone")
def_gdm_v1.createOrReplaceTempView("def_gdm_v1_vw")

In [0]:
# Apply split function in the required fields
def_gdm = split_multiple_delimiters(def_gdm_v1, "improvement_goal_concern", "improvement_goal_concern")
def_gdm = split_multiple_delimiters(def_gdm, "analysis_concern", "analysis_concern")
def_gdm = split_multiple_delimiters(def_gdm, "concern_zone", "concern_zone")

# def_gdm = split_multiple_delimiters(def_gdm, "purchase_channel", "purchase_channel")

def_gdm = explode_arrays(def_gdm) # explode first array levels

# Last view before unpivot
def_gdm.createOrReplaceTempView("def_gdm_vw")

In [0]:
###  UNPIVOT DEF_GDM
# 1. Identificar dinámicamente las columnas de atributos
non_attributes = ["source_name","brand_code","brand_country","registration_sub_source",'channel_name','created_date','brand_mdm_id','source_customer_id','new_client_key']
attributes = [c for c in def_gdm.columns if c not in non_attributes] 

# 2. Crear la expresión para el unpivot con stack()
num_atributos = len(attributes)
stack_expr = ", ".join(
    [f"'{col}', {col}" for col in attributes]
)

# 3. Realizar el unpivot usando `stack()`
gdm_unpivot = def_gdm.select(
    *non_attributes,
    F.expr(f"stack({num_atributos}, {stack_expr}) as (attribute, value)"))

# remove white spaces and capitalize values
gdm_unpivot = clean_text(gdm_unpivot, "value", "value_k")

# print(gdm_unpivot.count()) # 18084200
gdm_unpivot.createOrReplaceTempView("gdm_unpivot_vw")

# tmp = gdm_unpivot.select('value','value_k').distinct()
# display(tmp)

In [0]:
data = [
    ('analysis_concern','CONCERN'),
    ('concern_zone','ZONE'),
    ('desired_color_finish','HAIR_COLOR_FINISH'), 
    ('desired_cover','HAIR_COLOR_COVER'), 
    ('hair_care_product_used','HAIRCARE_PRODUCT'),
    ('improvement_goal_concern','CONCERN'),
    ('last_hair_color_service','HAIR_COLOR_SERVICE'),
    ('last_hair_style_look','HAIR_STYLE_LOOK'),
    ('left_eye_color','EYE_COLOR'),
    ('look_occasion_makeup','MAKEUP_LOOK_OCCASION'),
    ('natural_hair_color','HAIR_COLOR'),
    ('right_eye_color','EYE_COLOR'),
    ('skin_sensitivity_zone','ZONE'),
    ('skin_type_zone','ZONE')
]


# Crear el DataFrame
columns = ["attribute", "attribute_k"]
attributes_mapping = spark.createDataFrame(data, columns)
attributes_mapping.createOrReplaceTempView("attributes_mapping_vw")


In [0]:
bmdm_lkp = spark.table("prod_latam_catalog.crm_analytics.bmdm_mapping_lkp_table")
# bmdm_global = spark.table("prod_latam_catalog.crm_analytics.bmdm_mapping_lkp_table")

# some transformations
bmdm_lkp = bmdm_lkp.withColumn('reference_name', F.upper('reference_name'))
bmdm_lkp = clean_text(bmdm_lkp, "reference_value", "value_k")

bmdm_lkp.createOrReplaceTempView("bmdm_lkp_vw")
# display(bmdm_lkp.limit(5))

# tmp = bmdm_lkp.filter(F.col("reference_name").isin(['ROUTINE_GOAL','ANALYSIS_TYPE'])).\
#   select('reference_name','value_k','reference_value').distinct()
# display(tmp)


In [0]:
## CRUCE CON LA LKP
bmdm_services = spark.sql("""
with bmdm_lkp as (
  select distinct 
    reference_name,
    value_k,
    reference_value,
    schema_name,
    axe_profile,
    dimentions
  from bmdm_lkp_vw
),

merge_1 as (
  select
  a.*,
  --b.attribute_k,
  CASE WHEN b.attribute_k IS NULL THEN upper(a.attribute)
    ELSE b.attribute_k END as attribute_k
from gdm_unpivot_vw a
left join attributes_mapping_vw b
  on a.attribute = b.attribute
),

merge_2 as (
  select
    a.*,
    CASE WHEN c.value_N IS NULL THEN a.value_k
      ELSE c.value_N END as value_def,
    c.state
  from merge_1 a
  left join prod_latam_catalog.crm_analytics.bmdm_attributes_values_cleaning c
    on a.attribute_k = c.attribute
    and a.value_k = c.value
)--,

--merge_2 as (
select 
a.* except (state),
CASE WHEN (a.value IS NULL OR len(a.value) = 0) and state IS NULL THEN 'Not assignable'
  WHEN a.value IS NOT NULL AND c.value_k IS NOT NULL THEN 'Mapped'
  WHEN (a.value IS NOT NULL AND a.value RLIKE '[0-9]') and state IS NULL THEN 'Not assignable'
  ELSE 'Misclassified' END AS state,
  c.value_k as value_lkp,
  c.schema_name,
  c.axe_profile,
  c.dimentions
from merge_2 a
left join bmdm_lkp c
  on a.attribute_k = c.reference_name
  and a.value_def = c.value_k
--),
""")


#display(tmp)
bmdm_services.createOrReplaceTempView("bmdm_services_vw")

In [0]:
services_table_1 = bmdm_services.filter(F.col('state').isin(['Mapped','Assignable'])).\
    select('source_name','channel_name',"brand_code","brand_country","registration_sub_source",'brand_mdm_id','source_customer_id','created_date','attribute_k','value_def','schema_name','axe_profile','dimentions','new_client_key').\
    distinct().\
    withColumn("source_name", F.initcap("source_name")).\
    withColumn("registration_sub_source", F.initcap("registration_sub_source")).\
    withColumnRenamed("attribute_k","attribute").\
    withColumnRenamed("value_def","value").\
    withColumn('source_name',F.regexp_replace("source_name","Sitecore",'Demandware')).\
    withColumn('value',F.regexp_replace("value","Uv",'UV')).\
    withColumn('value',F.regexp_replace("value",'Seeking to update my routine ','Seeking to update my routine')).\
    withColumn('brand_code',F.regexp_replace("brand_code","LP",'LOP'))
    # withColumn("channel_name", F.initcap("channel_name")).\
    # withColumn('channel_name',F.regexp_replace("channel_name","Dsf Services",'DSF Services')).\


# remove white spaces, replace '_' and capitalize values
services_table_1 = clean_text(services_table_1, "attribute", "attribute")

services_table_1.createOrReplaceTempView("services_table_1_vw")
# print(services_table_1.count()) #1246302

In [0]:
##### FINAL TABLE  ####

# QUERY OFICIAL
query = f"""
with dim_brand as (
  SELECT DISTINCT
    brand_code,
    division, 
    FIRST_VALUE(brand_name) OVER (PARTITION BY brand_code ORDER BY brand_name desc) AS brand_name
  FROM crm_reporting.vw_dim_brand
),

--step needed to bring the brand_customer_id and subsequently, the buyer status
dim_customer_bridge as (
  select distinct
    brand_mdm_id,
    brand_customer_id
  from prod_latam_catalog.crm_reporting.dim_customer_bridge
),

fact_order_sales as (
  select distinct
    brand_customer_id
  from prod_latam_catalog.crm_reporting.fact_order_sales
),

fact_segment_by_brand as (
  select distinct
    brand_customer_id
  from prod_latam_catalog.crm_reporting.fact_segment_by_brand
  where contactability_key != 'CONNON'
)

select 
  b.division,
  b.brand_name,
  case 
    when registration_sub_source in ('Hair Quiz','Contest') then 'Hair Quiz'
    when registration_sub_source in ('Skin Dr','Quizzes/diagnostic Tools') then 'Skin Dr'
    when registration_sub_source in ('Product Finder') then 'Spotscan' END as service,
  a.*,
  case when d.brand_customer_id is null then 'No buyer' else 'Buyer' end as buyer_key,
  case when e.brand_customer_id is null then 'Not contactable' else 'Contactable' end as contactability_key
from services_table_1_vw a
left join dim_brand b
  on a.brand_code = b.brand_code
left join dim_customer_bridge c
  on a.brand_mdm_id = c.brand_mdm_id
left join fact_order_sales d
  on c.brand_customer_id = d.brand_customer_id
left join fact_segment_by_brand e
  on c.brand_customer_id = e.brand_customer_id
  
"""
#lifetime_buyer_activity_segment_key

services_table = spark.sql(query)
services_table.createOrReplaceTempView("services_table_vw")
# display(ids_faltantes)
# print(services_table.count()) #1246302

In [0]:
# Para crear una tabla permanente en el catalogo
services_table.write.mode('overwrite').option("mergeSchema", "true").format("delta").saveAsTable("prod_latam_catalog.crm_analytics.bmdm_services_attributes")

### VALIDATION

In [0]:
# 4. Agrupar y agregar: contar IDs, obtener fechas mínimas y máximas
df_result = bmdm_services.groupBy("source_name","brand_code","brand_country","registration_sub_source",'attribute','attribute_k','value','value_k','value_def','state','value_lkp').agg(
        F.countDistinct("brand_mdm_id").alias("id_counts"),
        F.min("created_date").alias("min_date"),
        F.max("created_date").alias("max_date")
    )

# Mostrar el resultado final
# print(df_result.count()) #2713

# tmp = df_result.filter(F.col("state").isNull())
tmp = df_result.filter(F.col("state") == 'Misclassified').\
  orderBy(['attribute_k','value'])
display(tmp)

tmp = df_result.select('attribute','attribute_k').distinct().\
  filter(F.col("attribute_k").isNull())
display(tmp)

source_name,brand_code,brand_country,registration_sub_source,attribute,attribute_k,value,value_k,value_def,state,value_lkp,id_counts,min_date,max_date
DEMANDWARE,KER,CHI,hair quiz,analysis_analysis_clinical_sign_sign_type,ANALYSIS_ANALYSIS_CLINICAL_SIGN_SIGN_TYPE,null,Null,Null,Misclassified,null,7,2025-01-22,2025-01-31
SITECORE,KER,ARG,hair quiz,analysis_analysis_clinical_sign_sign_type,ANALYSIS_ANALYSIS_CLINICAL_SIGN_SIGN_TYPE,null,Null,Null,Misclassified,null,2,2025-01-24,2025-01-27
DEMANDWARE,KER,BRA,hair quiz,analysis_analysis_clinical_sign_sign_type,ANALYSIS_ANALYSIS_CLINICAL_SIGN_SIGN_TYPE,null,Null,Null,Misclassified,null,13,2025-01-22,2025-01-30
JEBBIT,KER,BRA,product finder,analysis_analysis_clinical_sign_sign_type,ANALYSIS_ANALYSIS_CLINICAL_SIGN_SIGN_TYPE,null,Null,Null,Misclassified,null,2,2025-01-01,2025-01-13
DEMANDWARE,KER,COL,hair quiz,analysis_analysis_clinical_sign_sign_type,ANALYSIS_ANALYSIS_CLINICAL_SIGN_SIGN_TYPE,null,Null,Null,Misclassified,null,3,2025-01-24,2025-01-26
DEMANDWARE,KER,BRA,hair quiz,analysis_type,ANALYSIS_TYPE,null,Null,Null,Misclassified,null,13,2025-01-22,2025-01-30
DEMANDWARE,KER,COL,hair quiz,analysis_type,ANALYSIS_TYPE,null,Null,Null,Misclassified,null,3,2025-01-24,2025-01-26
SITECORE,KER,ARG,hair quiz,analysis_type,ANALYSIS_TYPE,null,Null,Null,Misclassified,null,2,2025-01-24,2025-01-27
JEBBIT,KER,BRA,product finder,analysis_type,ANALYSIS_TYPE,null,Null,Null,Misclassified,null,2,2025-01-01,2025-01-13
DEMANDWARE,KER,CHI,hair quiz,analysis_type,ANALYSIS_TYPE,null,Null,Null,Misclassified,null,7,2025-01-22,2025-01-31


attribute,attribute_k


In [0]:
# tmp = bmdm_services.filter((F.col("axe_profile").isNull()) &
#                            (F.col("state").isin(['Mapped','Assignable'])))
# display(tmp)

In [0]:
%sql
select *--count(distinct brand_mdm_id)
from services_table_vw
where brand_mdm_id = '000382e658c94dd225cfd365e91adbd3'

division,brand_name,service,source_name,channel_name,brand_code,brand_country,registration_sub_source,brand_mdm_id,source_customer_id,created_date,attribute,value,schema_name,axe_profile,dimentions,new_client_key,buyer_key,contactability_key
LUXE,Urban Decay,null,Demandware,DSF Services,UD,MEX,Сart Checkout,000382e658c94dd225cfd365e91adbd3,f883b6fd918f670fdaeecb847010f120,2023-02-24,Skin tone,Fair,Beauty Profile,Skin Profile,Description,New,No buyer,Contactable
LUXE,Urban Decay,null,Demandware,DSF Services,UD,MEX,Сart Checkout,000382e658c94dd225cfd365e91adbd3,f883b6fd918f670fdaeecb847010f120,2023-02-24,Skin undertone,Cool,Other,Other,Other,New,No buyer,Contactable
LUXE,Urban Decay,null,Demandware,DSF Services,UD,MEX,Сart Checkout,000382e658c94dd225cfd365e91adbd3,f883b6fd918f670fdaeecb847010f120,2023-02-24,Makeup look,Bold makeup,Beauty Profile,MakeUp Profile,Description,New,No buyer,Contactable


In [0]:
# # # conteo a partir de pais-marca-clave-valor
# tmp = bmdm_services.select('source_name',"brand_code","brand_country","registration_sub_source",'attribute_k','value_def','state').distinct().\
#   groupBy('state').agg(F.count("state").alias("state_counts"))

# display(tmp)

# # conteo a partir de clave-valor
# tmp = bmdm_services.select('attribute_k','value_def','state').distinct().\
#   groupBy('state').agg(F.count("state").alias("state_counts"))
  
# display(tmp)

In [0]:
summary = spark.sql(
    """
-- PRIMER CRUCE
select 
  source_name,
  brand_code,
  brand_country, 
  registration_sub_source,
  to_date(min(created_date)) min_date,
  to_date(max(created_date)) max_date,
  count(distinct brand_mdm_id) brand_mdm_id
from def_gdm_vw 
--where lower(registration_sub_source) IN ('skin dr','hair quiz','product finder')
group by all
order by brand_country, brand_code
    """)

#print(cruce.count())
display(summary)

source_name,brand_code,brand_country,registration_sub_source,min_date,max_date,brand_mdm_id
SITECORE,KER,ARG,hair quiz,2023-04-11,2025-03-12,7935
DEMANDWARE,DMC,BRA,skin dr,2024-03-13,2024-09-04,1031
JEBBIT,DMC,BRA,product finder,2025-02-13,2025-03-12,1584
DEMANDWARE,KER,BRA,hair quiz,2023-03-06,2025-03-12,18662
JEBBIT,KER,BRA,product finder,2024-12-18,2025-03-12,1938
JEBBIT,LAN,BRA,product finder,2024-10-07,2025-03-10,211
DEMANDWARE,LRP,BRA,skin dr,2024-10-14,2025-03-12,1286
DEMANDWARE,LRP,BRA,product finder,2023-04-14,2024-07-07,40704
JEBBIT,SKI,BRA,quizzes/diagnostic tools,2024-06-13,2025-03-12,3937
DEMANDWARE,VIC,BRA,skin dr,2024-07-23,2024-08-08,3


In [0]:
# %sql
# describe crm_reporting.fact_message_tracker

## UNIT TESTS

In [0]:
# %sql
# select created_dt
# -- from crm_reporting.dim_customer_bridge
# from prod_latam_catalog.crm_reporting.dim_gdm_brand_profile
# where brand_mdm_id = '80ed61e233bc6d0c9b653328c6f35950'

In [0]:
%sql
select *
from crm_reporting.dim_customer_bridge
-- from prod_latam_catalog.crm_reporting.dim_gdm_brand_profile
where brand_mdm_id = '0c347de61cde1d1234fdbbba0fe56f43'

brand_code,brand_country,source_name,bridge_customer_id,mdm_source,source_customer_id,customer_id,brand_customer_id,global_customer_id,brand_mdm_id,global_mdm_id,record_status,sys_last_modified_by,sys_last_modified_dt,etl_batch_id
LAN,CHI,DEMANDWARE,74d1d777e571f1b92088afbb27996cc3,DDM,861809d1b2ebbc96bd09cc2d6fb5ceaf,60527a054be0a2829a0d8aa941074b35,6b6b04126847938f9a19962b18812e7e,49f6aff6e5100956a0251aeea903f3fb,0c347de61cde1d1234fdbbba0fe56f43,def22b21e8f36e8deb115b80477655af,ACTIVE,spark_user,2024-02-29T01:32:54.264Z,1081_20240229010516


In [0]:
%sql
select brand_country,brand_code,source_name,source_customer_id,date(created_dt) as created_dt,date(last_modified_dt) as last_modified_dt
from prod_latam_catalog.crm_reporting.dim_customer 
where source_customer_id in ('861809d1b2ebbc96bd09cc2d6fb5ceaf') 
  and brand_country = 'CHI'  
  and brand_code = 'LAN'
order by created_dt

brand_country,brand_code,source_name,source_customer_id,created_dt,last_modified_dt
CHI,LAN,DEMANDWARE,861809d1b2ebbc96bd09cc2d6fb5ceaf,null,2024-02-28


In [0]:
# %sql
# select distinct * --analysis,concern_improvement_goal 
# --brand_mdm_id, count(brand_mdm_id) as counts
# from gdm_cols_filt_vw
# where brand_mdm_id = '044b4fb407f4702b749fb1f76cb2dc8d'
# -- where hair_type = 'Grosso'
# --'8473ba14591a9441ae2adf617f2af010' #49rows

In [0]:
# %sql
# select *--analysis,fragrance_routine --brand_mdm_id, count(brand_mdm_id) as counts
# from expld_gdm_vw
# -- from expld_gdm_vw 
# -- where brand_mdm_id = '00136159905ba5997705eec39c8e0fe8' -- no tiene axe ni dim
# where brand_mdm_id = '044b4fb407f4702b749fb1f76cb2dc8d' --explode 36 lineas
# -- where concern_zone like '%,%'

In [0]:
# %sql
# select distinct *--analysis,fragrance_routine --brand_mdm_id, count(brand_mdm_id) as counts
# from services_table_vw 
# where brand_mdm_id = '00136159905ba5997705eec39c8e0fe8' --explode 36 lineas  

In [0]:
# ## DETALLE DE DUPLICADOS
# tmp = spark.sql("""
# select brand_mdm_id, count(brand_mdm_id) as counts
# from expld_gdm_vw
# group by brand_mdm_id
# having counts > 1
# order by counts desc
# --limit 1000
# """)

# display(tmp)
# tmp.createOrReplaceTempView("tmp_vw")

In [0]:
# %sql
# -- valida cuantos brand_mdm_id tienen mas de un registro
# select counts, count(brand_mdm_id) as brand_mdm_id_counts
# from tmp_vw
# group by counts
# order by brand_mdm_id_counts desc

### Dim channel validation

In [0]:
# %sql
# SELECT snapshot_year_month, 
# sum(lifetime_consumer_cnt) FROM prod_latam_catalog.crm_reporting.vw_summary_segment_by_brand
# WHERE upper(acquisition_channel) like "%JEBBIT%" and snapshot_year_month = "20240630"
# GROUP BY ALL

In [0]:
# %sql
# SELECT * FROM prod_latam_catalog.crm_reporting.dim_channel
# 'hola'

In [0]:
# %sql
# SELECT snapshot_date_key, b.channel_name, b.row_key,
# count(distinct brand_customer_id) 
# FROM prod_latam_catalog.crm_reporting.fact_segment_by_brand a 
# INNER JOIN prod_latam_catalog.crm_reporting.dim_channel b ON a.acquisition_channel_key = b.row_key
# WHERE snapshot_date_key = "20240630" and
#       (acquisition_channel_key like '%DEMANDWARE%' or 
#        acquisition_channel_key like '%SITECORE%' or
#        acquisition_channel_key like '%JEBBIT%')
# GROUP BY ALL

In [0]:
# %sql
# SELECT snapshot_date_key, --b.channel_name,--row_key,
# count(distinct brand_customer_id) 
# FROM prod_latam_catalog.crm_reporting.fact_segment_by_brand a 
# INNER JOIN prod_latam_catalog.crm_reporting.dim_channel b ON a.acquisition_channel_key = b.row_key
# WHERE snapshot_date_key = "20240630" and
# /*
#       (acquisition_channel_key like '%DEMANDWARE%' or 
#        acquisition_channel_key like '%SITECORE%'
# */
#        acquisition_channel_key like '%JEBBIT%'
# GROUP BY ALL

In [0]:
# ### Merged tables
# merged_gdm = spark.sql("""
# select 
#   --a.is_deleted, 
#   e.channel_name,
#   upper(a.source_name) as source_name,
#   --lower(a.registration_sub_source) AS registration_sub_source, 
#   count(distinct c.brand_customer_id)
#   --a.created_dt as created_date,
#   --d.*
# from prod_latam_catalog.crm_reporting.dim_customer a
# -- inner join prod_latam_catalog.crm_analytics.dim_services_manual b
# --   on upper(a.source_name) = b.source_name 
# --   and a.brand_code = b.brand_code 
# --   and a.brand_country = b.brand_country
# inner join prod_latam_catalog.crm_reporting.dim_customer_bridge c
#   on a.source_customer_id = c.source_customer_id 
#   and a.brand_code = c.brand_code 
#   and a.brand_country = c.brand_country
# -- inner join expld_gdm_vw d 
# inner join gdm_cols_filt_vw d
#   on c.brand_mdm_id = d.brand_mdm_id 
#   and c.brand_code = d.brand_code 
#   and c.brand_country = d.brand_country
# left join prod_latam_catalog.crm_reporting.dim_channel e 
#   on concat(UPPER(a.source_name),"_",UPPER(a.acq_source),"_",UPPER(a.registration_source)) = e.row_key
# where upper(a.source_name) IN ('DEMANDWARE','JEBBIT','SITECORE') 
#   AND (lower(a.registration_source) LIKE ('%quiz%') 
#      OR lower(a.registration_source) LIKE ('%diagnos%')
#      OR lower(a.registration_source) LIKE ('%website%'))
#      AND coalesce(a.is_deleted,'N') <> "Y"
#   --AND lower(a.registration_sub_source) IN ('skin dr','hair quiz','product finder')
#   -- lower(a.registration_sub_source) not in ('contest',null,'registration','','Footer')
#   --AND to_date(a.created_dt) >= '2023-01-01'
# group by all
# --order by registration_sub_source,brand_code, brand_country

# """)

# display(merged_gdm)
# # print(merged_gdm.count()) #291103